# Airtel, Netflix, and Amazon Prime Video: Dataset and Model Workflow

This notebook preserves the existing Jio workflow and source CSVs. Airtel modeling is an educational exercise on the Airtel-folder data only: its customer IDs and status labels overlap with Jio, and provenance is unverified. Netflix is trained only after structural validation of the binary target. Amazon receives data-quality analysis only because no defensible churn target is available. Reports contain aggregate checks and no personal values.


In [1]:

from pathlib import Path
from io import BytesIO
import numpy as np
import pandas as pd
import joblib
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

ROOT = Path(r"C:\Users\mudda\OneDrive - BENNETT UNIVERSITY\Customer Churn Preddiction & Analytics Dashboard")
OUT = {
    "airtel_model": ROOT / "models" / "airtel_churn_model.pkl",
    "netflix_model": ROOT / "models" / "netflix_churn_model.pkl",
    "airtel_quality": ROOT / "reports" / "airtel_data_quality_summary.csv",
    "airtel_metrics": ROOT / "reports" / "airtel_model_comparison.csv",
    "netflix_metrics": ROOT / "reports" / "netflix_model_comparison.csv",
    "amazon_quality": ROOT / "reports" / "amazon_data_quality_summary.csv",
}
existing_outputs = [str(p.resolve()) for p in OUT.values() if p.exists()]
if existing_outputs:
    raise FileExistsError(
        "Refusing to overwrite existing workflow outputs. No files were written. "
        "Choose new output paths or explicitly preserve/archive the existing outputs first:\n"
        + "\n".join(existing_outputs)
    )
missing = [str(p) for p in OUT.values() if not p.exists()]
if missing:
    raise FileNotFoundError("Expected output destinations are unavailable: " + ", ".join(missing))

def preprocessor_for(X, scale_numeric):
    numeric = [c for c in X.columns if pd.api.types.is_numeric_dtype(X[c])]
    categorical = [c for c in X.columns if c not in numeric]
    numeric_steps = [("imputer", SimpleImputer(strategy="median"))]
    if scale_numeric:
        numeric_steps.append(("scaler", StandardScaler()))
    return ColumnTransformer([
        ("num", Pipeline(numeric_steps), numeric),
        ("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("encoder", OneHotEncoder(handle_unknown="ignore"))
        ]), categorical)
    ])

def fit_compare(dataset_name, data, target, dropped, target_names, model_path, report_path, note):
    X = data.drop(columns=sorted(set(dropped + [target])))
    y = data[target].astype(int)
    if y.isna().any() or y.nunique() != 2:
        raise ValueError(dataset_name + " target must be complete and binary.")
    counts = y.value_counts().sort_index()
    if counts.min() < 2:
        raise ValueError(dataset_name + " has too few records per class for a stratified split.")
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )
    minority_share = float(counts.min() / counts.sum())
    models = {
        "Logistic Regression": Pipeline([
            ("preprocessor", preprocessor_for(X, True)),
            ("classifier", LogisticRegression(max_iter=1000))
        ]),
        "Random Forest": Pipeline([
            ("preprocessor", preprocessor_for(X, False)),
            ("classifier", RandomForestClassifier(
                n_estimators=200, random_state=42,
                class_weight="balanced" if minority_share < 0.40 else None
            ))
        ])
    }
    rows = []
    fitted = {}
    for model_name, pipeline in models.items():
        pipeline.fit(X_train, y_train)
        prediction = pipeline.predict(X_test)
        class_report = classification_report(
            y_test, prediction, labels=[0, 1], target_names=target_names,
            output_dict=True, zero_division=0
        )
        cm = confusion_matrix(y_test, prediction, labels=[0, 1])
        row = {
            "dataset": dataset_name, "model": model_name,
            "accuracy": accuracy_score(y_test, prediction),
            "train_rows": len(X_train), "test_rows": len(X_test),
            "train_class_0": int((y_train == 0).sum()), "train_class_1": int((y_train == 1).sum()),
            "test_class_0": int((y_test == 0).sum()), "test_class_1": int((y_test == 1).sum()),
            "tn": int(cm[0, 0]), "fp": int(cm[0, 1]), "fn": int(cm[1, 0]), "tp": int(cm[1, 1]),
            "rf_class_weight_balanced": bool(model_name == "Random Forest" and minority_share < 0.40),
            "scope_note": note
        }
        for label in target_names:
            key = label.lower().replace(" ", "_")
            row[key + "_precision"] = class_report[label]["precision"]
            row[key + "_recall"] = class_report[label]["recall"]
            row[key + "_f1"] = class_report[label]["f1-score"]
            row[key + "_support"] = class_report[label]["support"]
        rows.append(row)
        fitted[model_name] = pipeline
        print(dataset_name, "|", model_name, "| accuracy", round(row["accuracy"], 4),
              "| churn recall", round(row[target_names[1].lower().replace(" ", "_") + "_recall"], 4),
              "| churn F1", round(row[target_names[1].lower().replace(" ", "_") + "_f1"], 4))
    churn_key = target_names[1].lower().replace(" ", "_")
    selected = max(
        rows, key=lambda r: (r[churn_key + "_f1"], r[churn_key + "_recall"], r["accuracy"])
    )
    selected_name = selected["model"]
    results = pd.DataFrame(rows)
    results["selected_model"] = results["model"].eq(selected_name)
    results["selection_rule"] = "Highest churn-class F1; tie-break by churn recall, then accuracy"
    buffer = BytesIO()
    joblib.dump(fitted[selected_name], buffer, compress=3)
    model_bytes = buffer.getvalue()
    sample = X_test.head(10).copy()
    expected = fitted[selected_name].predict(sample)
    loaded_memory = joblib.load(BytesIO(model_bytes))
    if not np.array_equal(loaded_memory.predict(sample), expected):
        raise AssertionError(dataset_name + " model failed in-memory reload prediction check.")
    return results, selected_name, model_bytes, sample, expected

def quality(dataset, check, field, value, note=""):
    quality_rows.append({
        "dataset": dataset, "check": check, "field": field, "value": value, "note": note
    })

# Load sources without changing them.
jio = pd.read_csv(ROOT / "Datasets" / "Jio" / "Jio_Customer_Data.csv")
jio.columns = [str(c).strip().lower() for c in jio.columns]
airtel = pd.read_csv(ROOT / "Datasets" / "Airtel" / "Main_churn_data_for_tableau.csv")
airtel_view = pd.read_csv(ROOT / "Datasets" / "Airtel" / "vw_ChurnData.csv")
airtel_predictions = pd.read_csv(ROOT / "Datasets" / "Airtel" / "Predictions.csv")
for frame in (airtel, airtel_view, airtel_predictions):
    frame.columns = [str(c).strip().lower() for c in frame.columns]

quality_rows = []
for label, frame in [
    ("Jio", jio), ("Airtel main", airtel),
    ("Airtel view", airtel_view), ("Airtel predictions", airtel_predictions)
]:
    quality(label, "rows", "", len(frame))
    quality(label, "duplicate_full_rows", "", int(frame.duplicated().sum()))
    if "customer_id" in frame.columns:
        quality(label, "duplicate_customer_ids", "customer_id",
                int(frame["customer_id"].duplicated().sum()))
    for column, missing in frame.isna().sum().items():
        if missing:
            quality(label, "missing_values", str(column), int(missing))

# Compare by ID and report only aggregate counts, never the IDs themselves.
jio_by_id = jio.set_index("customer_id").sort_index()
airtel_by_id = airtel.set_index("customer_id").sort_index()
shared = jio_by_id.index.intersection(airtel_by_id.index)
quality("Airtel vs Jio", "shared_customer_ids", "customer_id", len(shared),
        "IDs are not included in this report.")
common = [c for c in jio_by_id.columns if c in airtel_by_id.columns]
row_equal = np.ones(len(shared), dtype=bool)
for column in common:
    left = jio_by_id.loc[shared, column]
    right = airtel_by_id.loc[shared, column]
    if pd.api.types.is_numeric_dtype(left) and pd.api.types.is_numeric_dtype(right):
        equal = np.isclose(pd.to_numeric(left, errors="coerce"),
                           pd.to_numeric(right, errors="coerce"), equal_nan=True)
    else:
        equal = (left.fillna("<MISSING>").astype(str).to_numpy()
                 == right.fillna("<MISSING>").astype(str).to_numpy())
    row_equal &= equal
    differences = int((~equal).sum())
    if differences:
        quality("Airtel vs Jio", "different_values_by_shared_id", column, differences,
                "Count only; feature values and IDs are not exposed.")
quality("Airtel vs Jio", "exact_rows_across_common_columns", "",
        int(row_equal.sum()), "Rows aligned by customer ID.")
if "customer_status" in common:
    labels_equal = (
        jio_by_id.loc[shared, "customer_status"].fillna("<MISSING>").astype(str).to_numpy()
        == airtel_by_id.loc[shared, "customer_status"].fillna("<MISSING>").astype(str).to_numpy()
    )
    quality("Airtel vs Jio", "matching_status_labels_by_shared_id", "customer_status",
            int(labels_equal.sum()), "Compared by shared ID; IDs are not exposed.")
nonjoined = jio[jio["customer_status"].isin(["Stayed", "Churned"])]
view_shared = set(airtel_view["customer_id"]).intersection(set(nonjoined["customer_id"]))
quality("Airtel view vs Jio", "shared_nonjoined_customer_ids", "customer_id",
        len(view_shared), "The view contains Stayed and Churned records only.")
quality("Airtel predictions", "joined_status_rows", "customer_status",
        int(airtel_predictions["customer_status"].eq("Joined").sum()))
if "customer_status_predicted" in airtel_predictions.columns:
    quality("Airtel predictions", "predicted_churned_rows", "customer_status_predicted",
            int(airtel_predictions["customer_status_predicted"].eq("Churned").sum()),
            "Predictions.csv is not used for model training.")

# Validate both supervised targets before fitting either model.
if airtel["customer_status"].isna().any():
    raise ValueError("Airtel customer_status contains missing labels.")
airtel_counts = airtel["customer_status"].value_counts(dropna=False)
for value, count in airtel_counts.items():
    quality("Airtel main", "target_distribution", "customer_status=" + str(value), int(count))
negative_monthly = int((pd.to_numeric(airtel["monthly_charge"], errors="coerce") < 0).sum())
monthly_min = float(pd.to_numeric(airtel["monthly_charge"], errors="coerce").min())
quality("Airtel main", "negative_monthly_charge_rows", "monthly_charge", negative_monthly,
        "Values are retained unchanged; minimum is " + str(monthly_min) + ".")
quality("Airtel modeling", "excluded_columns", "",
        "customer_id; customer_status; churn_category; churn_reason; total_charges; total_refunds; total_extra_data_charges; total_long_distance_charges; total_revenue",
        "Direct targets and post-outcome fields are excluded. Lifetime aggregates are excluded because no prediction-time cutoff is documented.")

netflix = pd.read_csv(ROOT / "Datasets" / "Netflix" / "netflix_customer_churn.csv")
if "churned" not in netflix.columns:
    raise ValueError("Netflix target column churned is missing; training stopped.")
if netflix["churned"].isna().any() or set(pd.unique(netflix["churned"])) != {0, 1}:
    raise ValueError("Netflix target must contain only nonmissing 0 and 1; training stopped.")
if netflix["customer_id"].duplicated().any():
    raise ValueError("Netflix contains duplicate customer IDs; training stopped pending review.")
quality("Netflix", "target_binary_nonmissing", "churned", "passed",
        "Structural validation only; the label definition and provenance are unverified.")
for value, count in netflix["churned"].value_counts().sort_index().items():
    quality("Netflix", "target_distribution", "churned=" + str(value), int(count))

# Airtel model: educational exercise; exclude Joined and lifetime totals with unknown timing.
airtel_model_data = airtel[airtel["customer_status"].isin(["Stayed", "Churned"])].copy()
airtel_model_data["churn"] = airtel_model_data["customer_status"].map(
    {"Stayed": 0, "Churned": 1}
).astype(int)
airtel_note = (
    "Educational exercise on Airtel-folder data only. Customer IDs and status labels overlap with Jio; "
    "provenance is unverified, so this is not evidence of real-world Airtel churn prediction."
)
airtel_results, airtel_selected, airtel_bytes, airtel_sample, airtel_expected = fit_compare(
    "Airtel-folder dataset", airtel_model_data, "churn",
    ["customer_id", "customer_status", "churn_category", "churn_reason",
     "total_charges", "total_refunds", "total_extra_data_charges",
     "total_long_distance_charges", "total_revenue"],
    ["Stayed", "Churned"], OUT["airtel_model"], OUT["airtel_metrics"], airtel_note
)

# Netflix: structurally validated before training; provenance and feature timing remain unknown.
netflix_note = (
    "Educational dataset exercise only. Target is structurally binary and complete; "
    "provenance, churn definition, and timing of behavior fields are unverified."
)
netflix_results, netflix_selected, netflix_bytes, netflix_sample, netflix_expected = fit_compare(
    "Netflix-folder dataset", netflix, "churned", ["customer_id", "churned"],
    ["Not Churned", "Churned"], OUT["netflix_model"], OUT["netflix_metrics"], netflix_note
)

# Amazon: quality report only; do not infer churn from renewal preference or dates.
amazon = pd.read_csv(ROOT / "Datasets" / "Amazon" / "amazon_prime_users.csv")
amazon_rows = []
def amazon_quality(check, field, value, note=""):
    amazon_rows.append({
        "dataset": "Amazon Prime Video", "check": check, "field": field, "value": value, "note": note
    })
amazon_quality("rows", "", len(amazon))
amazon_quality("columns", "", len(amazon.columns))
amazon_quality("missing_cells", "", int(amazon.isna().sum().sum()))
amazon_quality("duplicate_full_rows", "", int(amazon.duplicated().sum()))
amazon_quality("supervised_churn_target", "", "not established",
               "Renewal Status describes preference/settings; dates alone do not establish churn.")
for value, count in amazon["Renewal Status"].value_counts(dropna=False).items():
    amazon_quality("renewal_status_count", str(value), int(count),
                   "Not treated as churn or retention.")
for column in ["Date of Birth", "Membership Start Date", "Membership End Date"]:
    parsed = pd.to_datetime(amazon[column], errors="coerce")
    amazon_quality("parseable_dates", column, int(parsed.notna().sum()))
    amazon_quality("unparseable_dates", column, int(parsed.isna().sum()))
start = pd.to_datetime(amazon["Membership Start Date"], errors="coerce")
end = pd.to_datetime(amazon["Membership End Date"], errors="coerce")
amazon_quality("end_before_start", "Membership dates", int((end < start).sum()),
               "Chronology check does not define churn.")
amazon_quality("pii_fields_excluded", "", "Name; Email Address; Username; Payment Information; User ID",
               "No personal values are included in the report.")
amazon_quality("model_created", "", "no", "No defensible churn target was found.")

# Preserve aggregate reports and selected complete pipelines; exclusive create prevents overwrites.
payloads = {
    OUT["airtel_model"]: airtel_bytes,
    OUT["netflix_model"]: netflix_bytes,
    OUT["airtel_quality"]: pd.DataFrame(quality_rows).to_csv(index=False).encode("utf-8"),
    OUT["airtel_metrics"]: airtel_results.to_csv(index=False).encode("utf-8"),
    OUT["netflix_metrics"]: netflix_results.to_csv(index=False).encode("utf-8"),
    OUT["amazon_quality"]: pd.DataFrame(amazon_rows).to_csv(index=False).encode("utf-8"),
}
existing_outputs = [str(path.resolve()) for path in payloads if path.exists()]
if existing_outputs:
    raise FileExistsError(
        "Refusing to overwrite existing workflow outputs. No files were written. "
        "Choose new output paths or explicitly preserve/archive the existing outputs first:\n"
        + "\n".join(existing_outputs)
    )
for path, payload in payloads.items():
    with path.open("xb") as output_file:
        output_file.write(payload)

for name, model_path, sample, expected in [
    ("Airtel-folder dataset", OUT["airtel_model"], airtel_sample, airtel_expected),
    ("Netflix-folder dataset", OUT["netflix_model"], netflix_sample, netflix_expected),
]:
    loaded = joblib.load(model_path)
    if not np.array_equal(loaded.predict(sample), expected):
        raise AssertionError(name + " saved model prediction check failed.")
    print(name, "| selected model saved and reloaded; prediction check passed on",
          len(sample), "held-out rows.")
print("Amazon data-quality report saved; no Amazon churn model was created.")


Airtel-folder dataset | Logistic Regression | accuracy 0.8186 | churn recall 0.6657 | churn F1 0.6794
Airtel-folder dataset | Random Forest | accuracy 0.7995 | churn recall 0.7147 | churn F1 0.673
Netflix-folder dataset | Logistic Regression | accuracy 0.887 | churn recall 0.9046 | churn F1 0.8895
Netflix-folder dataset | Random Forest | accuracy 0.983 | churn recall 0.9801 | churn F1 0.9831
Airtel-folder dataset | selected model saved and reloaded; prediction check passed on 10 held-out rows.
Netflix-folder dataset | selected model saved and reloaded; prediction check passed on 10 held-out rows.
Amazon data-quality report saved; no Amazon churn model was created.
